# Credit Risk Analysis & Machine Learning Pipeline
## Phase 2: Python Machine Learning (The AI Brain)
Objective: In the previous SQL phase, we cleaned the data, handled outliers, and engineered features like loan_to_income_pct and Expected Loss components.

In this phase, we will:

Load the SQL-cleaned dataset.
Prevent "Data Leakage" by removing post-outcome variables.
Translate text categories into numbers (One-Hot Encoding).
Train an XGBoost Machine Learning model to predict Probability of Default (PD).


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

print("✅ Libraries imported successfully! Ready to build the brain.")

✅ Libraries imported successfully! Ready to build the brain.


2.1 Load Cleaned Data
We will load the dataset we exported from PostgreSQL (cleaned_credit_data.csv). This data already has our engineered bands and Expected Loss math baked in.


In [3]:
# Load the cleaned data exported from PostgreSQL
df = pd.read_csv("cleaned_credit_data.csv")
# Display part of the Data
display(df.head())
print(f"\n Data shape {df.shape[0]} rows, {df.shape[1]} Column.")


,person_age,age_band,person_income,income_band,person_home_ownership,person_emp_length,employment_category,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,loan_to_income_pct,cb_person_default_on_file,cb_person_cred_hist_length,exposure_at_default,loss_given_default
0,23,Young_Adult,34800,Medium_Income,RENT,7,Mid_Employee,EDUCATION,A,2500,6.39,0,0.07,7.18,N,2,2500,0.6
1,26,Adult,50000,Medium_Income,OWN,6,Mid_Employee,EDUCATION,B,3000,9.91,0,0.06,6.00,N,2,3000,0.6
2,26,Adult,90000,High_Income,RENT,8,Mid_Employee,PERSONAL,C,7000,13.49,0,0.08,7.78,Y,3,7000,0.6
3,25,Adult,60000,Medium_Income,OWN,0,New_Employee,VENTURE,B,9000,9.91,0,0.15,15.00,N,3,9000,0.6
4,25,Adult,15600,Low_Income,RENT,6,Mid_Employee,MEDICAL,B,5000,10.75,1,0.32,32.05,N,3,5000,0.6



 Data shape 32581 rows, 18 Column.


## Block 3: Feature Engineering & Preprocessing
3.1 Preventing Data Leakage & Separating Target
We separate our Target (loan_status: 0=Good, 1=Default) from our Features (X), and drop the Expected Loss components (exposure_at_default, loss_given_default) to prevent data leakage 


In [4]:
# Separate the Target (y) from the Features (X)
y = df["loan_status"]
X = df.drop(["loan_status", "age_band", "income_band", "employment_category", "exposure_at_default", "loss_given_default"], axis= 1)
print("✅ Target separated and leakage prevented.")


✅ Target separated and leakage prevented.


3.2 One-Hot Encoding

Machine Learning models only understand numbers, not text like "RENT" or "EDUCATION". We use pd.get_dummies to convert categories into binary (0/1) columns.

We use drop_first=True to avoid the "Dummy Variable Trap" (multicollinearity), which can confuse the model's math.


In [5]:
# The Translator: One-Hot Encoding
# This turns our remaining text (like 'RENT' and 'EDUCATION') into 0s and 1s.
X_encoded = pd.get_dummies(X, drop_first= True)
print(f"Total clues (features) given to the AI: {X_encoded.shape[1]}")


Total clues (features) given to the AI: 23


## Block 4: Train/Test Split & XGBoost Model Training

4.1 Train/Test Split & Model Training (XGBoost)

80% Training Set: We feed this to the AI so it can learn the patterns of who defaults and who doesn't.
20% Test Set: We hide this from the AI. After it learns, we test it to see if it can predict the hidden defaults.
Handling Imbalanced Data:Our EDA showed we have way more Good Loans (0) than Defaults (1). To stop the AI from being Lazy and just guessing 0 every time, we use scale_pos_weight=3. This tells the AI to pay 3x more attention to the rare Default cases!





In [6]:
# Split the data (80% to teach the AI, 20% to test the AI)
# stratify=y ensures the 80/20 split maintains the exact same ratio of Good/Default loans
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size= 0.2, stratify=y)

xgb= XGBClassifier(scale_pos_weight=3, random_state=42)

print("Teaching AI....")
xgb.fit(X_train, y_train)

print("Testing the AI on new data...")
y_pred = xgb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n AI Accuracy: {accuracy * 100:.2f}%\n")

print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))


Teaching AI....
Testing the AI on new data...

 AI Accuracy: 92.57%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      5095
           1       0.86      0.79      0.82      1422

    accuracy                           0.93      6517
   macro avg       0.90      0.88      0.89      6517
weighted avg       0.92      0.93      0.92      6517



In [7]:
# 1. Get the Probabilities! 
# predict_proba gives us [Probability of 0, Probability of 1]
# We only want the probability of 1 (Default), so we grab column [:, 1]
probabilities = xgb.predict_proba(X_test)[:, 1]

# 2. Let's look at the first 5 probabilities!
print("First 5 Default Probabilities (PD):")
print(probabilities[:5])

# 3. Attach these probabilities to our Test Data to see them side-by-side
# We make a copy of our test set
results_df = X_test.copy()

# Add the actual answers and the AI's predicted probability
results_df['Actual_Default'] = y_test
results_df['AI_Predicted_PD'] = probabilities

# Display the results!
display(results_df[['person_age', 'loan_int_rate', 'Actual_Default', 'AI_Predicted_PD']].head(10))

First 5 Default Probabilities (PD):
[1.6407140e-01 7.6820500e-02 6.4704573e-04 3.8664427e-04 4.1665435e-01]


,person_age,loan_int_rate,Actual_Default,AI_Predicted_PD
4434,24,13.11,0,0.164071
18660,33,7.29,0,0.076821
16592,34,10.59,0,0.000647
18700,33,7.51,0,0.000387
20527,28,11.71,0,0.416654
7721,23,7.51,0,0.096374
27814,45,16.00,1,0.714163
26241,47,8.59,0,0.437693
10108,23,13.67,1,0.986235
9324,24,5.42,0,0.012379


In [8]:
# 1. Get probabilities for EVERYONE (the whole 32,581 people, not just the 20% test)
full_probabilities = xgb.predict_proba(X_encoded)[:, 1]

# 2. Add the PD to our original 'data' dataframe (which still has our SQL bands!)
df['AI_Predicted_PD'] = full_probabilities

# 3. INSTRUCTOR UPGRADE: Calculate Expected Loss (EL) in Python!
# EL = EAD (Exposure) * PD (Probability) * LGD (Loss Given Default)
df['Expected_Loss'] = df['exposure_at_default'] * df['AI_Predicted_PD'] * df['loss_given_default']

# 4. Save it for Power BI!
df.to_csv('powerbi_credit_risk_data.csv', index=False)

print("✅ Success! Data exported to 'powerbi_credit_risk_data.csv'")
print(f"Total Expected Loss for the bank: ${df['Expected_Loss'].sum():,.2f}")

# Let's look at the final masterpiece!
display(df[['person_age', 'age_band', 'loan_amnt', 'AI_Predicted_PD', 'Expected_Loss']].head(10))

✅ Success! Data exported to 'powerbi_credit_risk_data.csv'
Total Expected Loss for the bank: $55,195,463.37


,person_age,age_band,loan_amnt,AI_Predicted_PD,Expected_Loss
0,23,Young_Adult,2500,0.060653,90.979384
1,26,Adult,3000,0.000417,0.750641
2,26,Adult,7000,0.034050,143.009636
3,25,Adult,9000,0.000005,0.024855
4,25,Adult,5000,0.999679,2999.036908
5,25,Adult,8400,0.134037,675.546194
6,26,Adult,8550,0.225690,1157.787138
7,26,Adult,1200,0.016269,11.713924
8,23,Young_Adult,16000,0.021238,203.886366
9,25,Adult,8400,0.183856,926.634218


In [10]:
import joblib

# 1. Save the AI Brain to a file
joblib.dump(xgb, 'xgb_credit_model.pkl')

# 2. Save the exact column structure so the web app remembers the 23 clues!
import json
model_columns = X_encoded.columns.tolist()
with open('model_columns.json', 'w') as f:
    json.dump(model_columns, f)

print("✅ AI Brain and Columns saved successfully!")

✅ AI Brain and Columns saved successfully!
